# Train and prepare the DermReady demo

Run 00_environment_setup.ipynb and 02_prepare_data.ipynb first. All learned results here use train/validation only; test remains untouched. Checkpoints are private and are not committed. This notebook does not publish a share link.


In [ ]:
from pathlib import Path
import os, sys, subprocess
repo=Path('/content/DermReady_');os.chdir(repo)
subprocess.run(['git','pull','--ff-only'],check=True)
from google.colab import drive
drive.mount('/content/drive')
persistent=Path('/content/drive/MyDrive/DermReady')
processed=persistent/'processed';raw=Path('/content/dermready_scin_raw')
checkpoint=persistent/'checkpoints';reports=persistent/'reports'
assert (processed/'cases.jsonl').exists(), 'Run 02_prepare_data.ipynb first.'
checkpoint.mkdir(parents=True,exist_ok=True);reports.mkdir(parents=True,exist_ok=True)


In [ ]:
subprocess.run([sys.executable,'scripts/cache_images.py','--processed-dir',str(processed),'--raw-dir',str(raw),'--report-dir',str(reports)],check=True)
subprocess.run([sys.executable,'scripts/quality_baseline.py','--processed-dir',str(processed),'--raw-dir',str(raw),'--checkpoint-dir',str(checkpoint),'--report-dir',str(reports)],check=True)


In [ ]:
subprocess.run([sys.executable,'-m','src.train','--config','configs/experiment.yaml','--processed-dir',str(processed),'--raw-dir',str(raw),'--checkpoint-dir',str(checkpoint),'--report-dir',str(reports)],check=True)


In [ ]:
subprocess.run([sys.executable,'scripts/select_policy.py','--processed-dir',str(processed),'--raw-dir',str(raw),'--checkpoint-dir',str(checkpoint),'--report-dir',str(reports)],check=True)
subprocess.run([sys.executable,'-m','pytest','tests','-q'],check=True)
import json
print('Prepared artifacts:', checkpoint)
print(json.loads((checkpoint/'decision_policy.json').read_text())['preferred_model'])


## Run locally

Download best.pt, quality_baseline.json, and decision_policy.json from your own Drive checkpoint folder. Put them in the local repository’s checkpoints folder and run `python app.py`. The feature baseline may be selected over MobileNet based on validation; `python app.py --model cnn` selects the trained image model. Read docs/RUN_APP.md for handling and limitations.
